# Analysing an A/B test

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Build confidence intervals for conversion rates
- Run a two-proportion z-test
- See how peeking inflates false positives

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(7)

## 1. The experiment
Control converts at 10%, the variant at 11%, with 15,000 users each.

In [ ]:
nA = nB = 15_000
a = rng.random(nA) < 0.10
b = rng.random(nB) < 0.11
pA, pB = a.mean(), b.mean()
for name, p, n in [('A', pA, nA), ('B', pB, nB)]:
    se = np.sqrt(p * (1 - p) / n)
    print(name, round(p, 4), '95% CI', (round(p - 1.96*se, 4), round(p + 1.96*se, 4)))

## 2. Two-proportion z-test

In [ ]:
p_pool = (a.sum() + b.sum()) / (nA + nB)
se = np.sqrt(p_pool * (1 - p_pool) * (1/nA + 1/nB))
z = (pB - pA) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print('z', round(z, 2), 'p-value', round(p_value, 4))

## 3. Sample size
Rule of thumb: n per group ≈ 16·p(1−p)/δ².

In [ ]:
p, delta = 0.10, 0.01
print(int(16 * p * (1 - p) / delta**2))

## 4. Why peeking is dangerous
Both groups are identical (no real effect). We check every day and stop at the first p < 0.05.

In [ ]:
def peeking_trial(days=20, per_day=500):
    a, b = np.array([]), np.array([])
    for _ in range(days):
        a = np.append(a, rng.random(per_day) < 0.1); b = np.append(b, rng.random(per_day) < 0.1)
        pp = (a.sum() + b.sum()) / (len(a) + len(b)); se = np.sqrt(pp*(1-pp)*(2/len(a)))
        if se > 0 and 2*(1 - stats.norm.cdf(abs((b.mean()-a.mean())/se))) < 0.05: return True
    return False
print('false-positive rate with peeking:', np.mean([peeking_trial() for _ in range(400)]))   # far above 0.05

## Exercises
1. How many users per group do you need to detect +2 points on a 20% baseline?
2. Re-run section 4 checking only once at the end. What false-positive rate do you get?

In [ ]:
# your work here

## Solutions

In [ ]:
print(int(16 * 0.2 * 0.8 / 0.02**2))   # 6400